# Self-Distillation on Successful Rollouts — RECAP-aligned Policy Improvement

**Core idea**: Fine-tune the BC policy on its own successful rollouts only.
This is the A_pos-only special case of RECAP — the policy learns from its
best behavior, reducing variance on hard tasks without any advantage token
or prefix/suffix modification.

**Why this works where full RECAP didn't**:
- Uses SmolVLA's **native forward path** for training (no monkey-patching)
- No advantage token, no prefix modification, no KV cache contamination
- Inference path is **identical** to the original BC checkpoint
- Only successful trajectories → policy sharpens around what already works

**Pipeline**:
1. Load BC baseline + existing rollout dataset (100 episodes, 73 success / 27 failure)
2. Filter to success-only frames (A_pos episodes)
3. Fine-tune action expert using native flow-matching loss
4. Smoke test on 2 tasks × 3 episodes
5. Full eval on 10 tasks × 5 episodes

**Expected**: +2-5% over BC baseline (73% → 75-78%), concentrated on high-variance tasks.


## Step 1 — Environment

In [3]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
DRIVE_ROOT = "/mnt/rollouts"
os.environ["HF_HOME"]         = f"{DRIVE_ROOT}/hf_cache"
os.environ["HF_LEROBOT_HOME"] = f"{DRIVE_ROOT}/lerobot_cache"


In [1]:
!apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!pip install -q "lerobot[smolvla,libero]" wandb scikit-learn



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
print(f"CUDA: {torch.cuda.is_available()}  |  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")


CUDA: True  |  NVIDIA A100 80GB PCIe


## Step 2 — W&B setup

In [6]:
import wandb

wandb.login()

WANDB_PROJECT = "IDL_34_RECAP"
WANDB_ENTITY  = "idl_34"
WANDB_MODE    = "online"   # "online" | "offline" | "disabled"

print(f"W&B project: {WANDB_PROJECT}, mode: {WANDB_MODE}")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hbhagwat (idl_34) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B project: IDL_34_RECAP, mode: online


## Step 3 — Constants

In [7]:
from pathlib import Path
import numpy as np

ROLLOUT_DATASET_DIR = Path(DRIVE_ROOT) / "rollouts_with_labels"
DISTILL_DIR         = Path(DRIVE_ROOT) / "self_distill"
DISTILL_DIR.mkdir(parents=True, exist_ok=True)

BC_POLICY_REPO  = "HuggingFaceVLA/smolvla_libero"
LIBERO_SUITE    = "libero_spatial"

TASK_DESCRIPTIONS = {
    0: "pick up the black bowl between the plate and the ramekin and place it on the plate",
    1: "pick up the black bowl from table center and place it on the plate",
    2: "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    3: "pick up the black bowl next to the cookie box and place it on the plate",
    4: "pick up the black bowl next to the plate and place it on the plate",
    5: "pick up the black bowl next to the ramekin and place it on the plate",
    6: "pick up the black bowl on the cookie box and place it on the plate",
    7: "pick up the black bowl on the ramekin and place it on the plate",
    8: "pick up the black bowl on the stove and place it on the plate",
    9: "pick up the black bowl on the wooden cabinet and place it on the plate",
}

assert ROLLOUT_DATASET_DIR.exists(), f"Rollout dataset not found at {ROLLOUT_DATASET_DIR}"
print(f"ROLLOUT_DATASET_DIR = {ROLLOUT_DATASET_DIR}")
print(f"DISTILL_DIR         = {DISTILL_DIR}")


ROLLOUT_DATASET_DIR = /mnt/rollouts/rollouts_with_labels
DISTILL_DIR         = /mnt/rollouts/self_distill


## Step 4 — LIBERO config

In [8]:
import yaml
import libero

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(parents=True, exist_ok=True)
libero_data_root = Path(DRIVE_ROOT) / "libero_data"
libero_data_root.mkdir(parents=True, exist_ok=True)
pkg_dir = Path(libero.__file__).parent / "libero"
with open(libero_config_dir / "config.yaml", "w") as f:
    yaml.safe_dump({
        "benchmark_root": str(pkg_dir),
        "bddl_files":     str(pkg_dir / "bddl_files"),
        "init_states":    str(pkg_dir / "init_files"),
        "datasets":       str(libero_data_root),
        "assets":         str(pkg_dir / "assets"),
    }, f)
print("LIBERO config written.")


LIBERO config written.


## Step 5 — Load BC policy (no modifications)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks

device = torch.device("cuda")
print(f"Loading BC baseline from {BC_POLICY_REPO} ...")
policy = SmolVLAPolicy.from_pretrained(BC_POLICY_REPO).to(device)

TRAINABLE_KEYWORDS = (
    "action_in_proj", "action_out_proj",
    "action_time_mlp",
    "state_proj",
    "lm_expert",
)
for name, p in policy.named_parameters():
    p.requires_grad = any(k in name for k in TRAINABLE_KEYWORDS)

trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
total     = sum(p.numel() for p in policy.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M")

tokenizer  = policy.model.vlm_with_expert.processor.tokenizer
CHUNK_SIZE = policy.config.chunk_size
MAX_ACTION_DIM = policy.config.max_action_dim
print(f"CHUNK_SIZE={CHUNK_SIZE}, MAX_ACTION_DIM={MAX_ACTION_DIM}")
print("No advantage token. No prefix/suffix modification. Native SmolVLA.")


## Step 6 — Rollout dataset (success-only filter)

We load the full 100-episode rollout dataset, then filter to only the 73
successful episodes (A_pos = 1). The policy will be fine-tuned exclusively
on trajectories where it succeeded.


In [ ]:
import glob
import pandas as pd
from torch.utils.data import Dataset


def _unpack(x):
    out = []
    stack = [x]
    while stack:
        item = stack.pop()
        if isinstance(item, (list, tuple, np.ndarray)):
            stack.extend(reversed(list(item)) if hasattr(item, "__iter__") else [item])
        else:
            out.append(item)
    return out


class RolloutFrameDataset(Dataset):
    def __init__(self, root, label_filter=None):
        self.files = sorted(glob.glob(str(Path(root) / "data" / "chunk-000" / "file-*.parquet")))
        assert self.files, f"No parquet files in {root}/data/chunk-000/"
        self._labels, self._file_idx, self._row_idx, self._task_idx = [], [], [], []
        tasks_pq = Path(root) / "meta" / "tasks.parquet"
        tdf = pd.read_parquet(tasks_pq)
        self._tasks_map = dict(zip(tdf["task_index"].tolist(), tdf.index.tolist()))

        print(f"Indexing {len(self.files)} parquet files ...")
        total_skipped = 0
        for fi, f in enumerate(self.files):
            df = pd.read_parquet(f, columns=["advantage_label", "task_index"])
            for r in range(len(df)):
                lbl_raw = df["advantage_label"].iat[r]
                lbl = int(lbl_raw[0]) if hasattr(lbl_raw, "__len__") else int(lbl_raw)
                if label_filter is not None and lbl != label_filter:
                    total_skipped += 1
                    continue
                ti_raw = df["task_index"].iat[r]
                ti = int(ti_raw[0]) if hasattr(ti_raw, "__len__") else int(ti_raw)
                self._labels.append(lbl)
                self._task_idx.append(ti)
                self._file_idx.append(fi)
                self._row_idx.append(r)

        self._labels = np.array(self._labels, dtype=np.int64)
        self._task_idx = np.array(self._task_idx, dtype=np.int64)
        self._cache = {}
        filter_str = {None: "all", 1: "success-only", 0: "failure-only"}[label_filter]
        print(f"Loaded {len(self._labels)} frames ({filter_str}), skipped {total_skipped}")

    def __len__(self):
        return len(self._labels)

    def _load_file(self, fi):
        if fi in self._cache:
            return self._cache[fi]
        df = pd.read_parquet(self.files[fi],
                             columns=["observation.images.image",
                                      "observation.images.image2",
                                      "observation.state",
                                      "action"])
        if len(self._cache) >= 4:
            self._cache.pop(next(iter(self._cache)))
        self._cache[fi] = df
        return df

    def __getitem__(self, i):
        df = self._load_file(self._file_idx[i])
        r = self._row_idx[i]
        img1   = np.asarray(_unpack(df["observation.images.image"].iat[r]),  dtype=np.float32).reshape(3, 256, 256)
        img2   = np.asarray(_unpack(df["observation.images.image2"].iat[r]), dtype=np.float32).reshape(3, 256, 256)
        state  = np.asarray(_unpack(df["observation.state"].iat[r]), dtype=np.float32)
        action = np.asarray(_unpack(df["action"].iat[r]), dtype=np.float32)
        return {
            "observation.images.image":  torch.from_numpy(img1),
            "observation.images.image2": torch.from_numpy(img2),
            "observation.state":         torch.from_numpy(state),
            "action":                    torch.from_numpy(action),
            "task":                      self._tasks_map.get(int(self._task_idx[i]), ""),
        }


# Load SUCCESS-ONLY frames
success_ds = RolloutFrameDataset(ROLLOUT_DATASET_DIR, label_filter=1)
print(f"\nSuccess-only dataset: {len(success_ds)} frames")


## Step 7 — Native flow-matching loss (no advantage token)

This uses the **exact same loss function** as SmolVLA's original BC training:
flow-matching MSE on the velocity field. No modifications to the forward path.

The only difference from original BC training: we train on the policy's own
successful rollouts instead of expert demonstrations.


In [ ]:
import json, time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from lerobot.utils.constants import ACTION


def collate(bl):
    return {
        "observation.images.image":  torch.stack([b["observation.images.image"] for b in bl]),
        "observation.images.image2": torch.stack([b["observation.images.image2"] for b in bl]),
        "observation.state":         torch.stack([b["observation.state"] for b in bl]),
        "action":                    torch.stack([b["action"] for b in bl]),
        "task":                      [b["task"] for b in bl],
    }


def batch_to_device(batch, dev):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(dev, non_blocking=True)
    return batch


def native_flow_matching_loss(policy, batch):
    """Exact same loss as SmolVLA BC training. No modifications."""
    bs  = batch[ACTION].shape[0]
    dev = batch[ACTION].device

    images, img_masks = policy.prepare_images(batch)
    state   = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)

    tok = tokenizer(list(batch["task"]), padding="max_length",
                    max_length=policy.config.tokenizer_max_length,
                    truncation=True, return_tensors="pt")
    lang_tokens = tok["input_ids"].to(dev)
    lang_masks  = tok["attention_mask"].to(dev).bool()

    if actions.ndim == 2:
        actions = actions.unsqueeze(1).expand(-1, CHUNK_SIZE, -1)
    if actions.shape[-1] < MAX_ACTION_DIM:
        actions = F.pad(actions, (0, MAX_ACTION_DIM - actions.shape[-1]))

    noise = policy.model.sample_noise(actions.shape, dev)
    t     = policy.model.sample_time(bs, dev)
    x_t   = t[:, None, None] * noise + (1.0 - t[:, None, None]) * actions
    u_t   = noise - actions

    prefix_embs, prefix_pad, _ = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state)
    suffix_embs, suffix_pad, _ = policy.model.embed_suffix(x_t, t)

    pad = torch.cat([prefix_pad, suffix_pad], dim=1)
    p_len, s_len = prefix_pad.shape[1], suffix_pad.shape[1]
    att = torch.cat([torch.zeros(bs, p_len, dtype=torch.bool, device=dev),
                     torch.ones(bs, s_len, dtype=torch.bool, device=dev)], dim=1)
    att_2d  = make_att_2d_masks(pad, att)
    pos_ids = torch.cumsum(pad, dim=1) - 1

    (_, suffix_out), _ = policy.model.vlm_with_expert.forward(
        attention_mask=att_2d, position_ids=pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, suffix_embs],
        use_cache=False, fill_kv_cache=False)
    suffix_out = suffix_out[:, -CHUNK_SIZE:].to(dtype=torch.float32)
    v_t = policy.model.action_out_proj(suffix_out)

    real_dim = policy.config.action_feature.shape[0]
    loss = F.mse_loss(v_t[..., :real_dim], u_t[..., :real_dim])
    return loss


print("Native flow-matching loss function defined (no advantage token).")


## Step 8 — Self-distillation training

Fine-tune the action expert on success-only rollouts. Short training:
1000 steps at batch 16 (~20 min). We use a lower learning rate (5e-5)
to avoid drifting too far from the BC baseline — the goal is sharpening,
not relearning.


In [ ]:
STEPS      = 1000
BATCH_SIZE = 16
LR         = 5e-5       # conservative: half of BC's original 1e-4
SAVE_EVERY = 500
LOG_EVERY  = 25

wb_run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY, mode=WANDB_MODE,
    name="self_distill_success_only",
    config={
        "method": "self-distillation",
        "data": "success-only rollouts",
        "n_frames": len(success_ds),
        "lr": LR,
        "steps": STEPS,
        "batch": BATCH_SIZE,
        "model": "SmolVLA-LIBERO",
        "task_suite": "libero_spatial",
    },
    reinit=True,
)

dl = DataLoader(success_ds, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=4, pin_memory=True, drop_last=True,
                collate_fn=collate, persistent_workers=True, prefetch_factor=2)

optimizer = torch.optim.AdamW(
    [p for p in policy.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-10, betas=(0.9, 0.95))

policy.train()
history = []
running_loss = 0.0
step = 0
t0 = time.time()
pbar = tqdm(total=STEPS, desc="self-distill", dynamic_ncols=True)

while step < STEPS:
    for batch in dl:
        if step >= STEPS:
            break
        batch = batch_to_device(batch, device)
        loss = native_flow_matching_loss(policy, batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
        optimizer.step()

        running_loss += loss.item()
        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        if step % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            elapsed = (time.time() - t0) / 60
            tqdm.write(f"[step {step:5d}] loss={avg:.4f}  elapsed={elapsed:.1f}m")
            history.append((step, avg))
            wb_run.log({"step": step, "loss": avg, "elapsed_min": elapsed})
            running_loss = 0.0

        if step % SAVE_EVERY == 0:
            ck = DISTILL_DIR / f"step_{step:06d}"
            ck.mkdir(parents=True, exist_ok=True)
            policy.save_pretrained(ck)

pbar.close()
policy.save_pretrained(DISTILL_DIR)
with open(DISTILL_DIR / "loss_history.json", "w") as f:
    json.dump(history, f)

wb_run.summary["final_loss"] = history[-1][1] if history else 0.0
wb_run.summary["elapsed_min"] = (time.time() - t0) / 60
wb_run.finish()

print(f"\nDone. Saved to {DISTILL_DIR}")
print(f"Final loss: {history[-1][1]:.4f}")


## Step 9 — Training loss plot

In [ ]:
import matplotlib.pyplot as plt

hist = np.array(history)
plt.figure(figsize=(9, 4))
plt.plot(hist[:, 0], hist[:, 1], color="#2a9d8f", linewidth=2)
plt.xlabel("step"); plt.ylabel("flow-matching loss")
plt.title("Self-distillation training loss (success-only rollouts)")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(DISTILL_DIR / "training_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {DISTILL_DIR / 'training_loss.png'}")


## Step 10 — Eval setup

Standard eval infrastructure. No monkey-patching needed — the policy
is a native SmolVLA checkpoint, so inference works out of the box.


In [ ]:
from copy import deepcopy
import einops
from tqdm import trange
import lerobot.scripts.lerobot_eval as _le
from lerobot.envs.utils import preprocess_observation, add_envs_task, check_env_attributes_and_types
from lerobot.utils.constants import ACTION as _ACTION, OBS_STR
from lerobot.utils.utils import inside_slurm


def _patched_rollout(env, policy, env_preprocessor, env_postprocessor, preprocessor, postprocessor,
                     seeds=None, return_observations=False, render_callback=None):
    policy.reset()
    observation, info = env.reset(seed=seeds)
    if render_callback is not None:
        render_callback(env)
    all_observations, all_actions, all_rewards, all_successes, all_dones = [], [], [], [], []
    step = 0
    done = np.array([False] * env.num_envs)
    max_steps = env.call("_max_episode_steps")[0]
    progbar = trange(max_steps, desc=f"rollout <=({max_steps} steps)", disable=inside_slurm(), leave=False)
    check_env_attributes_and_types(env)
    while not np.all(done) and step < max_steps:
        observation = preprocess_observation(observation)
        observation = add_envs_task(env, observation)
        observation = env_preprocessor(observation)
        if return_observations:
            flat = {k: v for k, v in observation.items() if isinstance(v, torch.Tensor)}
            all_observations.append(deepcopy(flat))
        observation = preprocessor(observation)
        with torch.inference_mode():
            action = policy.select_action(observation)
        action = postprocessor(action)
        action_transition = {_ACTION: action}
        action_transition = env_postprocessor(action_transition)
        action = action_transition[_ACTION]
        action_numpy = action.to("cpu").numpy()
        observation, reward, terminated, truncated, info = env.step(action_numpy)
        if render_callback is not None:
            render_callback(env)
        successes = info["final_info"]["is_success"].tolist() if "final_info" in info else [False] * env.num_envs
        done = terminated | truncated | done
        if step + 1 == max_steps:
            done = np.ones_like(done, dtype=bool)
        all_actions.append(torch.from_numpy(action_numpy))
        all_rewards.append(torch.from_numpy(reward))
        all_dones.append(torch.from_numpy(done))
        all_successes.append(torch.tensor(successes))
        step += 1
        progbar.update()
    if return_observations:
        observation = preprocess_observation(observation)
        observation = add_envs_task(env, observation)
        observation = env_preprocessor(observation)
        flat = {k: v for k, v in observation.items() if isinstance(v, torch.Tensor)}
        all_observations.append(deepcopy(flat))
    ret = {
        _ACTION: torch.stack(all_actions, dim=1),
        "reward": torch.stack(all_rewards, dim=1),
        "success": torch.stack(all_successes, dim=1),
        "done": torch.stack(all_dones, dim=1),
    }
    if return_observations:
        stacked = {}
        for key in all_observations[0]:
            stacked[key] = torch.stack([obs[key] for obs in all_observations], dim=1)
        ret[OBS_STR] = stacked
    if hasattr(policy, "use_original_modules"):
        policy.use_original_modules()
    return ret

_le.rollout = _patched_rollout

from lerobot.envs.configs import LiberoEnv as LiberoEnvCfg
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.factory import make_pre_post_processors
from lerobot.scripts.lerobot_eval import eval_policy

_env_cfg_tmp = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=[0])
env_preprocessor, env_postprocessor = make_env_pre_post_processors(
    env_cfg=_env_cfg_tmp, policy_cfg=policy.config)
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=BC_POLICY_REPO,
    preprocessor_overrides={"device_processor": {"device": str(policy.config.device)}})
print("Eval infrastructure ready.")


## Step 11 — Smoke test: 2 tasks × 3 episodes

Quick sanity check before full eval. Run on task 0 (easy, BC ~80%)
and task 5 (hard, BC ~40%). If self-distilled policy matches or beats
BC on both, proceed to full eval.

**Critical**: if task 0 gives 0/3, training broke the policy — stop and
reload from a checkpoint. This should NOT happen because we used the
native forward path with no modifications.


In [ ]:
policy.eval()

env_cfg_smoke = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=[0, 5])
envs_smoke = make_env(env_cfg_smoke, n_envs=1, use_async_envs=False)

for task_id in [0, 5]:
    env = envs_smoke[LIBERO_SUITE][task_id]
    t0 = time.time()
    result = eval_policy(
        env=env, policy=policy,
        env_preprocessor=env_preprocessor, env_postprocessor=env_postprocessor,
        preprocessor=preprocessor, postprocessor=postprocessor,
        n_episodes=3, max_episodes_rendered=0, videos_dir=None,
        return_episode_data=False, start_seed=60000 + 1000 * task_id,
    )
    successes = [bool(ep["success"]) for ep in result["per_episode"]]
    elapsed = (time.time() - t0) / 60
    print(f"Task {task_id}: {sum(successes)}/3  ({elapsed:.1f} min)")

print("\nIf task 0 >= 2/3: proceed to full eval.")
print("If task 0 = 0/3: STOP — training broke the policy.")


## Step 12 — Full eval: 10 tasks × 5 episodes

Compare with BC baseline (from your existing measurements).
Use different seeds from the BC baseline eval to avoid overlap.


In [ ]:
N_EPISODES_PER_TASK = 5
TASK_IDS = list(range(10))

env_cfg_full = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=TASK_IDS)
envs_by_suite = make_env(env_cfg_full, n_envs=1, use_async_envs=False)
task_envs = envs_by_suite[LIBERO_SUITE]

# BC baseline numbers (from your 5-ep eval — the table in your slides)
BC_PER_TASK = {0: 80, 1: 60, 2: 80, 3: 60, 4: 60, 5: 20, 6: 60, 7: 40, 8: 80, 9: 80}

wb_run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY, mode=WANDB_MODE,
    name="self_distill_eval",
    config={"method": "self-distillation eval", "n_eps_per_task": N_EPISODES_PER_TASK},
    reinit=True,
)

policy.eval()
distill_per_task = {}
t0_eval = time.time()

for task_id in TASK_IDS:
    env = task_envs[task_id]
    result = eval_policy(
        env=env, policy=policy,
        env_preprocessor=env_preprocessor, env_postprocessor=env_postprocessor,
        preprocessor=preprocessor, postprocessor=postprocessor,
        n_episodes=N_EPISODES_PER_TASK,
        max_episodes_rendered=0, videos_dir=None,
        return_episode_data=False,
        start_seed=70000 + 1000 * task_id,
    )
    successes = [bool(ep["success"]) for ep in result["per_episode"]]
    sr = 100 * sum(successes) / len(successes)
    distill_per_task[task_id] = sr
    bc_sr = BC_PER_TASK[task_id]
    delta = sr - bc_sr
    marker = "+" if delta > 0 else ("=" if delta == 0 else "")
    print(f"  Task {task_id}: distill={sr:.0f}%  BC={bc_sr}%  ({marker}{delta:+.0f}%)")
    wb_run.log({f"task_{task_id}_distill": sr, f"task_{task_id}_bc": bc_sr, f"task_{task_id}_delta": delta})

distill_overall = np.mean(list(distill_per_task.values()))
bc_overall = np.mean(list(BC_PER_TASK.values()))
elapsed = (time.time() - t0_eval) / 60

print(f"\n{'='*50}")
print(f"OVERALL: Self-distill = {distill_overall:.1f}%  |  BC = {bc_overall:.1f}%  |  Δ = {distill_overall - bc_overall:+.1f}%")
print(f"Eval time: {elapsed:.1f} min")

wb_run.summary["overall_distill"] = distill_overall
wb_run.summary["overall_bc"] = bc_overall
wb_run.summary["overall_delta"] = distill_overall - bc_overall
wb_run.summary["eval_time_min"] = elapsed
wb_run.finish()


## Step 13 — Comparison histogram

In [ ]:
import matplotlib.pyplot as plt

task_labels = [f"T{i}" for i in range(10)]
bc_rates     = [BC_PER_TASK[i] for i in range(10)]
distill_rates = [distill_per_task[i] for i in range(10)]

x = np.arange(len(task_labels))
width = 0.38

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, bc_rates,      width, label="BC baseline",
               color="#6c757d", edgecolor="black", linewidth=0.6)
bars2 = ax.bar(x + width/2, distill_rates, width, label="Self-distilled",
               color="#2a9d8f", edgecolor="black", linewidth=0.6)

ax.set_xlabel("Task ID", fontsize=12)
ax.set_ylabel("Success rate (%)", fontsize=12)
ax.set_title("LIBERO-Spatial: BC baseline vs Self-distilled (5 episodes / task)", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(task_labels)
ax.set_ylim(0, 105)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
ax.legend(loc="upper right", frameon=True)

for rect in bars1:
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 1.5,
            f"{int(rect.get_height())}", ha="center", fontsize=9)
for rect in bars2:
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 1.5,
            f"{int(rect.get_height())}", ha="center", fontsize=9)

ax.text(0.02, 0.95, f"Overall:\nBC = {bc_overall:.0f}%\nDistill = {distill_overall:.0f}%",
        transform=ax.transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray"))

plt.tight_layout()
plt.savefig(DISTILL_DIR / "bc_vs_distill_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Save results
results = {
    "bc_per_task": BC_PER_TASK,
    "distill_per_task": {str(k): v for k, v in distill_per_task.items()},
    "bc_overall": bc_overall,
    "distill_overall": distill_overall,
}
with open(DISTILL_DIR / "eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved results to {DISTILL_DIR / 'eval_results.json'}")
